<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cst2312_h11/blob/main/CST2312_H11_Class25_pandasBasics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CST2312: Information and Data Management I**
## Class #11: Pandas Basics
### NYC Open Data, Board of Health Restaurant Inspections

**Week #06**

A copy of this workbook is available at [bit.ly/cst2312cl11](https://bit.ly/cst2312cl11)

*revisions: Apr 2022 | Dec 2022 | May 2023 | Apr 2024 | Dec 2025 | Apr 26 | May 26 | Oct 26 |*

**Author:** Professor Patrick

## Learning Objectives

By the end of this session, students will be able to:

1. **Load** a CSV file into a pandas DataFrame and **verify** the column data types.
2. **Convert** columns to numeric, datetime, and categorical types using `pd.to_numeric`, `pd.to_datetime`, and `pd.Categorical`.
3. **Identify** and **handle** missing values using `.isna()`, `.notna()`, `.dropna()`, and `.fillna()`.
4. **Select** rows from a DataFrame using both boolean indexing and the `.query()` method.
5. **Project**, **rename**, **sort**, and **deduplicate** columns using `.filter()`, `.rename()`, `.sort_values()`, and `.drop_duplicates()`.
6. **Compute** summary statistics using `.describe()`, `.value_counts()`, `.agg()`, and `.groupby()`.
7. **Produce** a horizontal bar chart from a DataFrame using native pandas plotting.

## External Reading Map

This notebook covers material outside the FOPP textbook. The following two open-access references are recommended:

| Source | When to Use |
|:---|:---|
| [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) (official user guide) | Quick reference and review of any single topic in this notebook. |
| [Python for Data Analysis, 3rd Edition](https://wesmckinney.com/book/) by Wes McKinney (open access) | Deeper treatment of any topic in this notebook. Particularly recommended chapters: 5 (Getting Started), 6 (Data Loading), 7 (Data Cleaning), 10 (Data Aggregation). |

A consolidated SQL-to-pandas reference table appears at the end of the CORE material in this notebook.

## Notebook Structure

This notebook is divided into two tiers:

* **[CORE]** sections cover the material intended for in-class delivery and form the basis for assessments.
* **[EXTENSION]** sections appear at the end of the notebook and provide depth, advanced patterns, and an engaging City Tech distance demonstration. Extension material is intended for self-study or for use in a follow-up session.

A short glossary of pandas terminology appears at the end of the notebook.

---
# **[CORE] Section 1: Setup and Preliminaries**

To read and process tabular data, this notebook uses three Python libraries:

* **pandas**: Provides the DataFrame and Series objects used throughout this notebook for tabular data.
* **numpy**: Provides numerical operations on arrays of values; pandas is built on top of numpy.
* **matplotlib**: Provides the plotting backend used by pandas for charts and figures.

The cell below imports each library under its conventional alias.

In [ ]:
# Data wrangling libraries
import pandas as pd      # DataFrame and Series functionality
import numpy as np       # Numerical operations on arrays

# Plotting library
import matplotlib        # Visualization (used implicitly by pandas plotting)

*Note: For richer statistical visualizations, the library **seaborn** may be imported as well. Seaborn is covered in a later session.*

---
# **[CORE] Section 2: Loading Data**

## Fetching the Dataset From NYC Open Data

This notebook uses the [DOHMH New York City Restaurant Inspection Results](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j) dataset, available from the City of New York.

Files fetched in a Google Colab notebook reside in the active Colab session and are removed when the session ends. The file is not transferred to the user's Google Drive.

The cell below uses the [Linux `curl` command](https://www.geeksforgeeks.org/curl-command-in-linux-with-examples/) to download the dataset.

>### Understanding the Shell Command
>
>The line below begins with an exclamation mark. In a Jupyter or Colab notebook, the exclamation mark tells the notebook to run the rest of the line as a shell command rather than as Python code. The command itself is `curl`, a Linux utility for transferring data from a URL.
>
>The `curl` command in this cell uses two pieces:
>
>* The **URL** in single quotes is the source: the NYC Open Data export endpoint for the inspection dataset.
>* The **`-o restaurant.csv`** flag tells `curl` to write the downloaded content to a local file named `restaurant.csv` rather than printing it to the screen.
>
>After this cell runs, a file named `restaurant.csv` is available in the working directory of the Colab session.

In [ ]:
# Fetches the most recent dataset from NYC Open Data
!curl 'https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD' -o restaurant.csv

## Reading the CSV Into a DataFrame

The pandas `read_csv` function reads a delimited text file and returns a DataFrame containing its rows and columns. The cell below creates a DataFrame variable named `restaurants` from the file just downloaded.

>### What the `read_csv()` Arguments Mean
>
>#### `"restaurant.csv"`
>The first positional argument: the file path or filename of the CSV to be read. Pandas looks for this file in the current working directory.
>
>#### `encoding="utf_8"`
>Specifies the character encoding used to decode the file. UTF-8 supports the full range of Unicode characters and is suitable for files containing international characters, accented letters, or special symbols. The strings `"utf_8"` and `"utf-8"` are accepted as equivalent.
>
>#### `dtype=str`
>Forces every column to be read as a string rather than allowing pandas to infer types automatically. This is useful when:
>
>* Preserving leading zeros (for example, ZIP codes or phone numbers)
>* Preventing pandas from misinterpreting mixed-type columns
>* Ensuring downstream code can rely on consistent string operations
>* Avoiding silent conversion errors during initial loading
>
>The trade-off is that numeric and date operations cannot be performed until the relevant columns are explicitly converted later. That conversion is the subject of Section 4 below.
>
>#### `low_memory=False`
>Controls how pandas reads the file into memory. By default (`low_memory=True`), pandas processes the file in chunks to reduce memory usage, which can produce mixed-type inference warnings (`DtypeWarning`) when a column contains values of inconsistent types across chunks. Setting `low_memory=False` causes pandas to read the entire file into memory at once, allowing it to determine column types from the complete dataset. This eliminates the warning at the cost of higher memory consumption, which is acceptable for moderately sized files such as a typical restaurant inspection dataset.

In [ ]:
restaurants = pd.read_csv(
    "restaurant.csv",
    encoding="utf_8",
    dtype=str,
    low_memory=False,
)

In [ ]:
len(restaurants)

## Inspecting the First Rows

The `.head(n)` method returns the first `n` rows of a DataFrame. With no argument, it returns five rows by default. This is the standard way to confirm that a file has been read correctly.

In [ ]:
restaurants.head()

The `.read_csv()` method has many additional options. See the [official documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) for the complete list.

---
# **[CORE] Section 3: Initial Exploration**

Before any analysis or transformation, it is essential to understand the structure of the DataFrame: which columns exist, what types they currently hold, and what the values look like.

## The `.columns` Attribute

The `.columns` attribute returns the index of column labels for the DataFrame. While it resembles a list, it is actually a pandas `Index` object, which provides additional functionality for label-based lookup.

In [ ]:
restaurants.columns

In [ ]:
type(restaurants.columns)

The columns can be iterated like any sequence:

In [ ]:
for column_name in restaurants.columns:
    print(column_name)

## Single-Column Access (Series)

A single column of a DataFrame is a pandas `Series`. Two equivalent forms of access are available:

* **Bracket notation**: `restaurants["ZIPCODE"]` (works for any column name)
* **Dot notation**: `restaurants.ZIPCODE` (works only when the column name contains no spaces or special characters and does not collide with a DataFrame attribute)

Both forms return the same Series object. The bracket form is more general and is preferred when there is any ambiguity.

In [ ]:
restaurants.ZIPCODE.tail()

*Note that the display of index numbers and values is followed by the line* `dtype: object`*. Pandas is reporting the data type of the Series.*

### Common Data Types

| Pandas dtype | Meaning |
|:---|:---|
| `object` | Text strings (or mixed types) |
| `int64` | 64-bit integers |
| `float64` | 64-bit floating-point numbers |
| `bool` | Boolean values (`True`/`False`) |
| `datetime64[ns]` | Timestamps with nanosecond precision |
| `category` | Categorical data with a fixed set of values |

*Let us inspect another column, the* `SCORE` *column, using bracket notation and `.tail(5)` for the last five values.*

In [ ]:
restaurants["SCORE"].tail(10)

❓ Why did the `SCORE` column not appear when `.head()` was called earlier?

💡 The DataFrame has many columns; the default display truncates the middle. The `.columns` attribute lists every column name regardless of how the DataFrame is displayed.

## Initial Summary with `.describe()`

The `.describe()` method produces a quick overview of the data in the DataFrame. Because every column is currently of dtype `object`, the output reports counts, unique values, the most frequent value (`top`), and that value's frequency. After type conversion in the next section, `.describe()` will produce richer numeric statistics.

In [ ]:
restaurants.describe()

The `.T` attribute transposes the result so that columns become rows, which is often easier to read when there are many columns:

In [ ]:
restaurants.describe().T

## Examining Column Data Types

The `.dtypes` attribute reports the current data type of each column.

In [ ]:
restaurants.dtypes

<font color=blue>The pandas dtype <b>object</b> refers to string or mixed content. To work with some of these values as dates or numbers, the columns must first be converted. Section 4 covers that conversion.</font>

---
# **[CORE] Section 4: Type Conversion**

Reading the file with `dtype=str` produced a DataFrame in which every column is a string. To perform numeric operations, date arithmetic, or memory-efficient categorical analysis, columns must be converted to appropriate types.

Pandas provides three primary conversion functions, each demonstrated below.

## Converting to Numeric

The `pd.to_numeric()` function converts a Series to numeric (integer or float, as appropriate). The cell below converts the `SCORE`, `Latitude`, and `Longitude` columns.

In [ ]:
restaurants["SCORE"] = pd.to_numeric(restaurants["SCORE"])
restaurants["Latitude"] = pd.to_numeric(restaurants["Latitude"])
restaurants["Longitude"] = pd.to_numeric(restaurants["Longitude"])
restaurants.dtypes

After conversion, the `SCORE` column is `float64` rather than `object`. Floating-point rather than integer is used because pandas represents missing values in numeric columns as `NaN`, which is a floating-point value.

In [ ]:
restaurants["SCORE"].tail(15)

## Converting to Datetime

The `pd.to_datetime()` function converts a Series of date-like strings to the `datetime64[ns]` dtype.

In [ ]:
restaurants["GRADE DATE"] = pd.to_datetime(restaurants["GRADE DATE"])
restaurants["RECORD DATE"] = pd.to_datetime(restaurants["RECORD DATE"])
restaurants["INSPECTION DATE"] = pd.to_datetime(restaurants["INSPECTION DATE"])
restaurants.dtypes

#### Note
In tricky cases, the `format` parameter can be used to specify the expected date format directly. See the [Python `strftime` documentation](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior) for the full list of format codes.

## Converting to Categorical Using a Dictionary Loop

A categorical column has a fixed, finite set of valid values. Categorical conversion is appropriate for columns such as borough, grade, violation code, and inspection type. Categorical columns use less memory than equivalent string columns and produce cleaner output in plots and pivot tables.

Rather than writing seven near-identical conversion statements, the cell below uses a dictionary that maps each column name to its conversion options. A single loop applies each conversion in turn. This pattern is common in pandas code and reinforces dictionary use from earlier in the course.

In [ ]:
# Each entry maps a column name to the keyword arguments for pd.Categorical
categorical_specs = {
    "BORO":                {"ordered": False},
    "GRADE":                {"categories": ["A", "B", "C"], "ordered": True},
    "VIOLATION CODE":       {"ordered": False},
    "CRITICAL FLAG":        {"ordered": False},
    "ACTION":               {"ordered": False},
    "CUISINE DESCRIPTION":  {"ordered": False},
    "INSPECTION TYPE":      {"ordered": False},
}

for column, spec in categorical_specs.items():
    restaurants[column] = pd.Categorical(restaurants[column], **spec)

restaurants.dtypes

The `GRADE` column receives the additional argument `categories=["A", "B", "C"]` and `ordered=True`, which tells pandas that the grades have a meaningful order. This is important for charts and for any future correlation analysis.

---
# **[CORE] Section 5: Descriptive Statistics by Type**

The `.describe()` method produces different output depending on the data type of the column it is called on. The three subsections below illustrate the three most common cases.

## Numeric Columns

For numeric columns, `.describe()` returns the eight standard summary statistics.

In [ ]:
restaurants["SCORE"].describe()

In [ ]:
restaurants["CUISINE DESCRIPTION"].describe()

>### The Eight Standard Rows of `.describe()` for Numeric Columns
>
>#### `count`
>The number of non-null values in the column. If `count` is smaller than the total number of rows, the column contains missing values.
>
>#### `mean`
>The arithmetic average of the non-null values. Sensitive to extreme values.
>
>#### `std`
>The standard deviation, which measures how spread out the values are around the mean. A small value indicates the data clusters tightly around the mean; a large value indicates wide dispersion.
>
>#### `min`
>The smallest non-null value in the column. Useful for spotting impossible or unexpected values.
>
>#### `25%` (First Quartile)
>The value below which 25 percent of the non-null observations fall.
>
>#### `50%` (Median)
>The middle value when the non-null observations are sorted. Half the values fall below it and half above. Unlike the mean, the median is not affected by extreme values.
>
>#### `75%` (Third Quartile)
>The value below which 75 percent of the non-null observations fall.
>
>#### `max`
>The largest non-null value in the column.

## Datetime Columns

For datetime columns, `.describe()` returns count, the most frequent value, and how often it appears, together with the minimum and maximum dates.

In [ ]:
restaurants[["INSPECTION DATE", "GRADE DATE", "RECORD DATE"]].describe()

>### Statistics Provided by `.describe()` for Datetime Columns
>
>#### `count`
>The number of non-null values in the column.
>
>#### `mean`
>The average datetime value (the midpoint of the time range, weighted by frequency).
>
>#### `min` and `max`
>The earliest and latest non-null dates in the column.
>
>#### `50%`
>The median date.

## Categorical and String Columns

For categorical or string columns, the more useful summary tool is `.value_counts()`, which returns a Series listing each unique value together with the number of times it appears.

In [ ]:
restaurants["BORO"].value_counts()

In [ ]:
restaurants["CUISINE DESCRIPTION"].value_counts().head(15)

The result of `.value_counts()` is itself a Series and can be sliced or assigned to a variable for reuse:

---
# **[CORE] Section 6: Missing Value Handling**

Real datasets routinely contain missing values. The NYC restaurant inspection data is no exception: not every record has a score, a grade, or geographic coordinates.

Pandas represents a missing value as `NaN` (Not a Number) for numeric columns and as `NaT` (Not a Time) for datetime columns. Both behave similarly for the purposes of detection and removal.

This section introduces the four most common methods for working with missing values.

## Detecting Missing Values: `.isna()` and `.notna()`

The `.isna()` method returns a boolean Series of the same length as the original, with `True` wherever a value is missing. The `.notna()` method is the logical opposite.

Both methods can be combined with `.sum()` to count occurrences, because Python treats `True` as `1` and `False` as `0`.

In [ ]:
# How many SCORE values are missing?
restaurants["SCORE"].isna().sum()

In [ ]:
# How many SCORE values are present?
restaurants["SCORE"].notna().sum()

In [ ]:
# Missing-value count for every column
restaurants.isna().sum()

## Removing Rows With Missing Values: `.dropna()`

The `.dropna()` method returns a new DataFrame with rows containing missing values removed. The `subset` parameter restricts the check to specific columns; without it, any missing value in any column causes the row to be dropped.

In [ ]:
# Drop rows where Latitude or Longitude is missing
restaurants_with_coords = restaurants.dropna(subset=["Latitude", "Longitude"])
print(f"Rows before drop: {len(restaurants):,}")
print(f"Rows after drop:  {len(restaurants_with_coords):,}")

## Filling Missing Values: `.fillna()`

The `.fillna()` method returns a new Series or DataFrame with missing values replaced by a specified value. The replacement should be chosen carefully to suit the analytical context.

In [ ]:
# Replace missing scores with the mean score (illustrative; not always appropriate)
mean_score = restaurants["SCORE"].mean()
score_filled = restaurants["SCORE"].fillna(mean_score)

# Compare missing counts before and after
print(f"Missing before fillna: {restaurants['SCORE'].isna().sum():,}")
print(f"Missing after fillna:  {score_filled.isna().sum():,}")

>#### Footnote: Why `NaN != NaN`
>The `NaN` value in pandas follows the IEEE 754 floating-point standard, which specifies that `NaN` is never equal to anything, including itself. This means an expression such as `Longitude == Longitude` evaluates to `False` precisely on the rows where `Longitude` is missing, and `True` on every other row. Some older pandas tutorials use this trick as an idiomatic test for "is not null". Modern pandas code uses `.notna()` instead because the intent is clearer.

---
# **[CORE] Section 7: Selecting Rows: Two Approaches**

Pandas provides two principal ways to select rows from a DataFrame based on a condition. Both produce equivalent results in most cases; they differ in syntax and in which is more readable for which kind of expression. Students should be fluent in both.

## Approach 1: Boolean Indexing

Boolean indexing uses a Series of `True` and `False` values to select rows: rows where the value is `True` are kept, rows where the value is `False` are dropped.

The condition is written using ordinary Python comparison operators applied to a column.

In [ ]:
# Select all rows in Brooklyn
brooklyn = restaurants[restaurants["BORO"] == "BROOKLYN"]
brooklyn.head()

In [ ]:
# Select all "Has Mice" violations (code 04L)
mice_violations = restaurants[restaurants["VIOLATION CODE"] == "04L"]
mice_violations.head()

## Approach 2: The `.query()` Method

The `.query()` method accepts a string expression that pandas evaluates against each row. It often reads more like English and is easier to chain with other methods.

In [ ]:
# The same Brooklyn selection using .query()
brooklyn = restaurants.query('BORO == "BROOKLYN"')
brooklyn.head()

### Backticks for Column Names With Spaces

The `.query()` string is parsed as a Python expression, which means column names with spaces or special characters cannot be used directly. To reference such a column, surround its name with backticks (the character `` ` ``, found on the same key as the tilde).

In [ ]:
# The "VIOLATION CODE" column contains a space, so backticks are required
has_mice = restaurants.query('`VIOLATION CODE` == "04L"')
has_mice.head()

## Combining Conditions

Multiple conditions are combined with `&` (AND) and `|` (OR). Each individual condition must be wrapped in parentheses when using boolean indexing because of Python's operator precedence rules.

```
# AND in pandas is "&"
# OR  in pandas is "|"
```

In [ ]:
# Boolean indexing: mice violations in Brooklyn
brooklyn_mice = restaurants[
    (restaurants["BORO"] == "BROOKLYN") &
    (restaurants["VIOLATION CODE"] == "04L")
]
brooklyn_mice.head()

In [ ]:
# Equivalent .query() form, more compact
brooklyn_mice = restaurants.query(
    ' BORO == "BROOKLYN" & `VIOLATION CODE` == "04L" '
)
brooklyn_mice.head()

<font color=red> <b>🚩 Alert!</b>  
While using the backtick ("~") is useful when dealing with blank spaces in columns names, it is best-practice to avoid the use of spaces in column names.  If column names have spaces in them, then dot-notation can't be used to reference them.  Another important stumbling block to avoid is using blank spaces at the start of a column name -- you will not be able to see the leading blank spaces and can waste a lot of time debugging.  Some datasets in .CSV files may have first lines that include column names but precede each name with a blank space.  The Pandas `read_csv()` method will include those leading blank spaces in column names.  For example, the following line in a .CSV file:</font>  
   
```  
id, city, state, population   

```  
<font color=red>Will result in `read_csv()` producing the following column names:</font>     
```  
["id", " city", " state", " population"]    
```  
<font color=red>That creates a dataframe with enigmatic column names for 'id', 'city', and 'state'.  
  
<font color=gray><i>See the Colab notebook at </i>[bit.ly/annoying_df](https://bit.ly/annoying_df)<i> for a working example of the frustration leading blanks can cause in column names.</i></font>  

## Frequency Analysis on a Filtered DataFrame

Once a filtered DataFrame is in hand, `.value_counts()` answers familiar questions such as "which restaurants appear most often in this subset?".

In [ ]:
# Top 20 restaurant names among "Has Mice" violations citywide
has_mice["DBA"].value_counts()[:20]

In [ ]:
# Restaurants with the most "Has Mice" violations, by their unique CAMIS identifier
has_mice["CAMIS"].value_counts()[:10]

---
# **[CORE] Section 8: Projection, Renaming, Sorting, and Deduplication**

## Projecting Columns With `.filter()`

Where boolean indexing and `.query()` select rows, `.filter()` selects columns. The `items` parameter accepts a list of column names to keep.

In [ ]:
restaurants.filter(items=["DBA", "GRADE", "GRADE DATE"]).head()

The `like` parameter selects all columns whose name contains a given substring:

In [ ]:
# All columns whose name contains "DATE"
restaurants.filter(like="DATE").head()

## Renaming Columns With `.rename()`

The `.rename()` method maps old column names to new ones using a dictionary. This is the pandas equivalent of SQL's `SELECT col AS alias`.

**Important:** `.rename()` returns a new DataFrame; it does not modify the original. The result must be assigned to a variable to be retained.

In [ ]:
# This call returns a new DataFrame; the result must be captured
restaurants2 = restaurants.rename(
    columns={
        "CAMIS":    "RESTAURANT_ID",
        "DBA":      "RESTAURANT_NAME",
        "BUILDING": "BUILDING_NUMBER",
        "BORO":     "BOROUGH",
    }
)
restaurants2.columns

## Sorting With `.sort_values()`

The `.sort_values()` method orders the rows by one or more columns. The `ascending` parameter accepts a single boolean or a list of booleans (one per sort column).

In [ ]:
# Sort by inspection date, oldest first
restaurants.sort_values("INSPECTION DATE", ascending=True).head(5)

In [ ]:
# Sort by inspection date (newest first), then by restaurant name (A to Z)
restaurants.sort_values(
    ["INSPECTION DATE", "DBA"],
    ascending=[False, True]
).head(5)

## Deduplicating With `.drop_duplicates()`

The `.drop_duplicates()` method returns a DataFrame with duplicate rows removed. By default it considers all columns; the `subset` parameter restricts the comparison to specific columns.

In [ ]:
# Distinct restaurant identifiers, names, boroughs, and addresses
unique_restaurants = (
    restaurants
    .filter(items=["CAMIS", "DBA", "BORO", "BUILDING", "STREET", "ZIPCODE"])
    .drop_duplicates()
)
print(f"Total inspection records: {len(restaurants):,}")
print(f"Unique restaurants:        {len(unique_restaurants):,}")

## Chain Notation

When several pandas operations are applied in sequence, the result of one becomes the input of the next. Chain notation expresses this sequence by enclosing the chained calls in parentheses and putting each call on its own line. The result is more readable than nested parentheses or many intermediate variables.

The example below combines filtering, deduplication, and sorting into a single readable expression.

In [ ]:
# Distinct Brooklyn restaurants with a "Has Mice" violation, sorted by name
(
    restaurants
    .query(' BORO == "BROOKLYN" & `VIOLATION CODE` == "04L" ')
    .filter(items=["DBA", "BUILDING", "STREET", "ZIPCODE"])
    .drop_duplicates()
    .sort_values("DBA")
    .head(15)
)

---
# **[CORE] Section 9: Basic Aggregation and Grouping**

Aggregation collapses many values into a single summary value. Grouping splits a DataFrame into subsets and applies aggregation to each subset independently. Together they answer questions of the form "for each X, what is the Y?".

The brief introductions below establish the concepts. Section 10 in the Extension material provides depth.

## Aggregation With `.agg()`

The `.agg()` method (also written as `.aggregate()`) computes one or more summary statistics on a Series or DataFrame.

In [ ]:
# A single statistic on a single column returns a single number
restaurants["SCORE"].agg("mean")

In [ ]:
# A list of statistics returns a Series
restaurants["SCORE"].agg(["mean", "std", "count", "min", "max"])

## Grouping With `.groupby()`

The `.groupby()` method splits the DataFrame into subsets based on the values of a column, then applies an aggregation to each subset.

In [ ]:
# Average inspection score by borough
restaurants.groupby("BORO", observed=True)["SCORE"].mean()

In [ ]:
# Multiple statistics per borough using named aggregations
(
    restaurants
    .groupby("BORO", observed=True)
    .agg(
        mean_score      = ("SCORE", "mean"),
        median_score    = ("SCORE", "median"),
        inspection_count = ("CAMIS", "count"),
        unique_restaurants = ("CAMIS", "nunique"),
    )
)

*The `observed=True` argument is included because `BORO` is a categorical column. Without it, current pandas versions emit a deprecation warning.*

---
# **SQL-to-Pandas Reference**

The table below maps common SQL operations to their pandas equivalents. Bookmark this section for quick reference later.

| SQL Operation | Pandas Equivalent |
|:---|:---|
| `SELECT col1, col2 FROM t` | `df.filter(items=["col1", "col2"])` or `df[["col1", "col2"]]` |
| `SELECT * FROM t WHERE col = 'x'` | `df.query("col == 'x'")` or `df[df["col"] == "x"]` |
| `SELECT * FROM t WHERE a = 1 AND b = 2` | `df.query("a == 1 & b == 2")` or `df[(df["a"]==1) & (df["b"]==2)]` |
| `SELECT DISTINCT col FROM t` | `df["col"].drop_duplicates()` or `df.drop_duplicates(subset=["col"])` |
| `SELECT * FROM t ORDER BY col DESC` | `df.sort_values("col", ascending=False)` |
| `SELECT col AS alias FROM t` | `df.rename(columns={"col": "alias"})` |
| `SELECT COUNT(*) FROM t` | `len(df)` or `df.shape[0]` |
| `SELECT AVG(col) FROM t` | `df["col"].mean()` |
| `SELECT col, COUNT(*) FROM t GROUP BY col` | `df.groupby("col").size()` |
| `SELECT col, AVG(other) FROM t GROUP BY col` | `df.groupby("col")["other"].mean()` |
| `SELECT * FROM t WHERE col IS NULL` | `df[df["col"].isna()]` |
| `SELECT * FROM t WHERE col IS NOT NULL` | `df[df["col"].notna()]` |

---
# **Exercise: Neighborhood Health Report**

You are a public-information analyst for a community board. The board has asked for a one-page health report covering restaurant inspection activity in a single ZIP code, so that residents can understand what restaurants in their neighborhood look like in the city's inspection records.

This exercise produces that report in six stages. Each stage has:

* A short description of what the stage produces.
* A **skeleton** code cell with a `TODO` comment showing where to write your code.
* A **verification** cell with `assert` statements that check the structure of your result.
* A collapsible **solution** block. Try each stage on your own first; expand the solution only after a serious attempt.

The recommended starting ZIP is **11201** (Downtown Brooklyn near City Tech). You may substitute any New York City ZIP code; the assertions are written to work with any ZIP that exists in the dataset.

> **Note on live data:** Because the dataset is fetched live from NYC Open Data, exact counts will vary from session to session. The verification cells check the *shape* and *type* of your result, not specific numeric values.

## Stage 1: Filter the Dataset to a Single ZIP Code

Produce a DataFrame named `neighborhood` containing every inspection record whose `ZIPCODE` matches `target_zip`.

In [ ]:
# TODO: replace the value below with the ZIP code you want to investigate.
# The recommended starter value is "11201" (Downtown Brooklyn near City Tech).
target_zip = "11201"

# TODO: filter the restaurants DataFrame to rows where ZIPCODE equals target_zip.
# Store the result in a DataFrame named `neighborhood`.
neighborhood = ...


In [ ]:
# Stage 1 verification
assert isinstance(neighborhood, pd.DataFrame), "neighborhood must be a DataFrame"
assert len(neighborhood) > 0, "neighborhood is empty: check the target_zip value"
assert (neighborhood["ZIPCODE"] == target_zip).all(), "all ZIPCODE values must equal target_zip"
print(f"OK: {len(neighborhood):,} inspection records in ZIP {target_zip}.")

<details>
<summary><b>Solution: Stage 1</b></summary>

```python
target_zip = "11201"
neighborhood = restaurants[restaurants["ZIPCODE"] == target_zip]
```

The boolean-indexing form is shown above. The equivalent `.query()` form would be:

```python
neighborhood = restaurants.query("ZIPCODE == @target_zip")
```

Note the `@target_zip` syntax inside `.query()`, which references a Python variable rather than a column name.

</details>

## Stage 2: Summary Statistics for the Neighborhood

Compute three summary numbers about `neighborhood`:

* `total_inspections`: the total number of inspection records.
* `unique_restaurants`: the number of distinct restaurants (use the `CAMIS` column, which is the restaurant identifier).
* `average_score`: the average inspection score across all rows in `neighborhood`.

In [ ]:
# TODO: count the total number of inspection records in `neighborhood`.
total_inspections = ...

# TODO: count the number of unique restaurants (use the CAMIS column).
unique_restaurants = ...

# TODO: compute the average inspection score in `neighborhood`.
average_score = ...

print(f"Total inspections:    {total_inspections:,}")
print(f"Unique restaurants:   {unique_restaurants:,}")
print(f"Average score:        {average_score:.2f}")

In [ ]:
# Stage 2 verification
assert isinstance(total_inspections, (int, np.integer)), "total_inspections must be an integer"
assert total_inspections > 0
assert isinstance(unique_restaurants, (int, np.integer)), "unique_restaurants must be an integer"
assert unique_restaurants > 0
assert unique_restaurants <= total_inspections, "unique restaurants cannot exceed total inspections"
assert isinstance(average_score, (float, np.floating)), "average_score must be a float"
assert average_score >= 0
print("OK: Stage 2 passed.")

<details>
<summary><b>Solution: Stage 2</b></summary>

```python
total_inspections = len(neighborhood)
unique_restaurants = neighborhood["CAMIS"].nunique()
average_score = neighborhood["SCORE"].mean()
```

Three notes:

* `len(df)` returns the number of rows in a DataFrame.
* `.nunique()` on a Series returns the count of distinct non-null values.
* `.mean()` automatically skips `NaN` values, so missing scores are excluded from the average.

</details>

## Stage 3: Grade Distribution

Produce a Series named `grade_counts` showing how many inspection records in `neighborhood` carry each grade (A, B, C, or other).

In [ ]:
# TODO: compute the count of each GRADE value in `neighborhood`.
# Store the result as a Series named `grade_counts`.
grade_counts = ...

grade_counts

In [ ]:
# Stage 3 verification
assert isinstance(grade_counts, pd.Series), "grade_counts must be a Series"
assert len(grade_counts) > 0, "grade_counts is empty"
assert any(g in grade_counts.index for g in ["A", "B", "C"]), \
    "grade_counts should contain at least one of A, B, or C"
print("OK: Stage 3 passed.")

<details>
<summary><b>Solution: Stage 3</b></summary>

```python
grade_counts = neighborhood["GRADE"].value_counts()
```

The result will likely include `NaN` counts as well. By default, `.value_counts()` excludes missing values; pass `dropna=False` to include them in the count.

</details>

## Stage 4: Top Five Violation Codes

Produce a Series named `top_violations` containing the five most frequent `VIOLATION CODE` values in `neighborhood`. The result will be ordered from most to least frequent.

In [ ]:
# TODO: produce the top 5 most frequent VIOLATION CODE values in `neighborhood`.
# Store the result as a Series named `top_violations`.
top_violations = ...

top_violations

In [ ]:
# Stage 4 verification
assert isinstance(top_violations, pd.Series), "top_violations must be a Series"
assert len(top_violations) <= 5, "top_violations must contain at most 5 entries"
if len(top_violations) > 1:
    sorted_descending = (top_violations.values[:-1] >= top_violations.values[1:]).all()
    assert sorted_descending, "top_violations must be in descending order of count"
print("OK: Stage 4 passed.")

<details>
<summary><b>Solution: Stage 4</b></summary>

```python
top_violations = neighborhood["VIOLATION CODE"].value_counts().head(5)
```

Slicing with `[:5]` would also work; `.head(5)` is the more idiomatic choice and reads more naturally.

</details>

## Stage 5: Ten Restaurants With the Most Inspection Records

Produce a Series named `top_restaurants` containing the ten restaurant names (`DBA` column) that appear most often in `neighborhood`.

In [ ]:
# TODO: produce the 10 most frequent DBA values in `neighborhood`.
# Store the result as a Series named `top_restaurants`.
top_restaurants = ...

top_restaurants

In [ ]:
# Stage 5 verification
assert isinstance(top_restaurants, pd.Series), "top_restaurants must be a Series"
assert len(top_restaurants) <= 10, "top_restaurants must contain at most 10 entries"
if len(top_restaurants) > 1:
    sorted_descending = (top_restaurants.values[:-1] >= top_restaurants.values[1:]).all()
    assert sorted_descending, "top_restaurants must be in descending order of count"
print("OK: Stage 5 passed.")

<details>
<summary><b>Solution: Stage 5</b></summary>

```python
top_restaurants = neighborhood["DBA"].value_counts().head(10)
```

</details>

## Stage 6: Horizontal Bar Chart of the Top Ten Restaurants

Produce a horizontal bar chart of `top_restaurants`. Reverse the order so that the restaurant with the most records appears at the top of the chart, which is the conventional reading direction for ranked horizontal bars.

This stage has no automated verification; inspect the chart visually.

In [ ]:
# TODO: produce a horizontal bar chart of top_restaurants.
# Hint: use .plot(kind="barh"). To put the largest bar at the top, reverse the
# Series first using slicing notation [::-1].
...

<details>
<summary><b>Solution: Stage 6</b></summary>

```python
top_restaurants[::-1].plot(kind="barh", title=f"Most-Inspected Restaurants in ZIP {target_zip}")
```

The `[::-1]` slice reverses the Series so that the largest count appears at the top of the chart. The `title` argument sets a chart title; this is optional but recommended for any chart that will be shared.

</details>

---
# **[EXTENSION] Section 10: Multi-Statistic and Named Aggregations**

Section 9 introduced `.agg()` with a single statistic on a single column. The same method scales to multiple statistics on multiple columns, and supports named aggregations that produce custom output column labels. The four examples below demonstrate the progression.

## Example 1: A Single Statistic on One Column

A single statistic name passed as a string produces a scalar value.

In [ ]:
restaurants["SCORE"].agg("mean")

## Example 2: Multiple Statistics on One Column

A list of statistic names produces a Series with one row per statistic.

In [ ]:
restaurants["SCORE"].agg(["mean", "std", "count", "nunique"])

## Example 3: Multiple Statistics on Multiple Columns (Dictionary Form)

A dictionary mapping column names to lists of statistics produces a DataFrame in which rows are statistics and columns are the original column names. Cells where a statistic does not apply are filled with `NaN`.

In [ ]:
restaurants.agg({
    "SCORE": ["mean", "std", "count", "nunique"],
    "CAMIS": ["nunique", "count"],
})

## Example 4: Named Aggregations Across Multiple Columns

Named aggregations use keyword arguments. Each keyword defines an output column with a custom name; each value is a tuple of `(source_column, statistic)`. This form is useful when the output should carry descriptive labels rather than the default statistic names.

In [ ]:
restaurants.agg(
    num_scored_violations = ("SCORE", "count"),
    mean_score            = ("SCORE", "mean"),
    std_score             = ("SCORE", "std"),
    num_entries           = ("CAMIS", "count"),
    num_restaurants       = ("CAMIS", "nunique"),
)

### Return Type Reference

| Call Pattern | Return Type |
|:---|:---|
| Series with a single statistic (string) | Scalar value (`float`, `int`) |
| Series with a list of statistics | Series |
| DataFrame with a single statistic (string) | Series |
| DataFrame with a list of statistics | DataFrame |
| DataFrame with a dictionary | DataFrame |
| DataFrame with named aggregations | DataFrame |

---
# **[EXTENSION] Section 11: Pivot Tables and Resampling**

[Pivot tables](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html) are one of the most flexible exploratory tools in pandas. They let a single function call answer questions of the form "for each combination of X and Y, what is the aggregate of Z?".

## Single-Dimension Pivot

The example below counts how many inspections occur on each grading date.

In [ ]:
pivot = pd.pivot_table(
    data=restaurants,
    index="GRADE DATE",     # specifies the rows
    values="CAMIS",         # specifies the content of the cells
    aggfunc="count",        # count distinct CAMIS values
)
pivot.head(10)

## Changing Date Granularity With `.resample()`

When a DataFrame has a datetime index, `.resample()` regroups the data into a different frequency, such as weekly or monthly intervals. It is the time-series counterpart of `.groupby()`.

The line below regroups daily counts into weekly totals and shows the most recent 100 weeks.

In [ ]:
pivot.resample("1W").sum().tail(100)

### Common `.resample()` Frequency Codes

The codes shown below are the current pandas conventions. Older codes such as `"M"` and `"Y"` still work but produce deprecation warnings in current pandas; the unambiguous forms are listed.

| Code | Meaning |
|:---|:---|
| `"D"` | Calendar day |
| `"B"` | Business day (weekdays only) |
| `"W"` | Weekly (ending Sunday by default) |
| `"W-MON"` | Weekly, ending Monday (any weekday substitutable) |
| `"ME"` | Month end |
| `"MS"` | Month start |
| `"QE"` | Quarter end |
| `"QS"` | Quarter start |
| `"YE"` | Year end |
| `"YS"` | Year start |
| `"h"` | Hourly |
| `"min"` | Minute |
| `"s"` | Second |

### Multipliers

A number placed in front of any code sets a custom interval:

| Code | Meaning |
|:---|:---|
| `"2D"` | Every 2 days |
| `"1W"` | Every 1 week |
| `"3ME"` | Every 3 months (quarterly) |
| `"6ME"` | Every 6 months (semi-annual) |
| `"15min"` | Every 15 minutes |

## Pivot Table With Two Dimensions

Adding a `columns` parameter breaks down the result by a second dimension. The example below counts inspections by date and borough.

In [ ]:
pivot2 = pd.pivot_table(
    data=restaurants,
    index="INSPECTION DATE",
    columns="BORO",
    values="CAMIS",
    aggfunc="count",
    observed=True,
)
pivot2.head(10)

### Removing Noise Rows and Columns

The dataset contains a placeholder date `1900-01-01` and a borough value `"0"` that are not real data. Both can be removed with `.drop()`.

In [ ]:
# axis="index" (or axis=0) drops a row by its index value
pivot2 = pivot2.drop(pd.to_datetime("1900-01-01"), axis="index", errors="ignore")

# axis="columns" (or axis=1) drops a column by name
pivot2 = pivot2.drop("0", axis="columns", errors="ignore")

pivot2.tail(5)

## Advanced Pivot Tables

Pivot tables can carry multiple aggregation functions and multi-level indexes and columns. The example below breaks down inspections by `BORO` and `GRADE`, and reports both the total count and the count of unique restaurants in each cell.

In [ ]:
# Multi-level columns: outer level is the statistic, inner level is BORO/GRADE
pivot_advanced = pd.pivot_table(
    data=restaurants,
    index="GRADE DATE",
    columns=["BORO", "GRADE"],
    values="CAMIS",
    aggfunc=["count", "nunique"],
    observed=True,
)

# Resample to monthly totals and transpose for easier reading
agg = pivot_advanced.resample("1ME").sum()
agg.tail().T

---
# **[EXTENSION] Section 12: Derived Columns with `.assign()` and `.apply()`**

The two methods covered in this section both produce new columns from existing ones, but they operate at different levels:

* **`.assign()`** is best when the new column can be expressed as a vectorized operation across whole columns of the DataFrame.
* **`.apply()`** is best when the new column requires a function that processes one row at a time (for example, a function that calls an external library expecting individual values rather than arrays).

Both are illustrated below using a recurring example: computing the distance between each restaurant in the dataset and the City Tech campus.

## Using the `.assign()` Approach

The function below computes the *Euclidean distance* (the straight-line "as the crow flies" distance) from each restaurant to City Tech. It is a rough approximation of the true geographic distance, but it is fast because the calculation is vectorized over the entire `Latitude` and `Longitude` columns at once.

In [ ]:
def euclidean_distance(df):
    """Compute the approximate distance in miles from City Tech to each row's
    Latitude/Longitude. The normalizing constants convert latitude and longitude
    differences to miles for the New York City area.
    """
    CityTech_lat = 40.6973
    CityTech_lon = -73.9861

    distance_squared = (
        ((df.Latitude  - CityTech_lat) / 0.0146) ** 2 +
        ((df.Longitude - CityTech_lon) / 0.0196) ** 2
    )
    return np.sqrt(distance_squared)


def combine_address(df):
    """Combine BUILDING, STREET, and ZIPCODE into a single uppercase address string."""
    return (df.BUILDING + " " + df.STREET + ", " + df.ZIPCODE).str.upper()

In [ ]:
# Use .assign() to create two new columns at once
(
    restaurants
    .assign(
        distance_from_CityTech = euclidean_distance,
        address                = combine_address,
    )
    .filter(items=["DBA", "address", "distance_from_CityTech"])
    .head()
)

In [ ]:
# Distinct restaurants nearest to City Tech, sorted by distance
(
    restaurants
    .assign(
        distance_from_CityTech = euclidean_distance,
        address                = combine_address,
    )
    .filter(items=["DBA", "address", "distance_from_CityTech"])
    .query("distance_from_CityTech > 0")    # eliminates NaN values
    .drop_duplicates()
    .sort_values("distance_from_CityTech")
    .head(20)
)

## Using the `.apply()` Approach

The Euclidean calculation above is fast because it operates on whole columns at once. Some calculations cannot be expressed that way: the `geopy` library, for example, accepts a single coordinate pair at a time, not a column of pairs. For that case, `.apply()` runs a function on each row in turn.

### Stage 1: Install and Import `geopy`

In [ ]:
!pip install -q -U geopy

from geopy import distance

>#### Why the Install and Import Are Needed
>
>##### `!pip install -q -U geopy`
>
>The `geopy` library is not part of Python's standard library and is not pre-installed in every environment. This command installs it.
>
>The flags have specific purposes:
>
>* `!` runs the command as a shell command rather than as Python code.
>* `-q` (quiet) suppresses most of the installation output to keep the notebook clean.
>* `-U` (upgrade) ensures the latest version is installed if an older version is already present.
>
>##### `from geopy import distance`
>
>Once `geopy` is installed, its functionality must be imported before it can be used. Without this line, Python would raise a `NameError` when the function below is called.

### Stage 2: Define the Per-Row Distance Function

The function below takes a single row of a DataFrame, extracts its `Latitude` and `Longitude`, and returns the geodesic distance in miles from City Tech.

In [ ]:
def distance_from_CityTech_geodesic(row):
    """Geodesic distance in miles from City Tech to the row's coordinates.

    Unlike euclidean_distance() above, this function operates on individual rows
    rather than whole columns. It is therefore used with .apply() rather than
    with .assign().
    """
    CityTech = (40.6973, -73.9861)
    rest     = (row.Latitude, row.Longitude)
    return distance.distance(CityTech, rest).miles

### Stage 3: Build a Smaller Working DataFrame

The full DataFrame contains many rows that share the same `(Latitude, Longitude)` pair (a single restaurant has many inspection records). To avoid wasted computation, the cell below builds a smaller DataFrame containing one row per unique restaurant location, with rows that have missing coordinates removed using `.dropna()`.

In [ ]:
rest_names_locations = (
    restaurants
    .assign(address=combine_address)
    .filter(items=["CAMIS", "DBA", "address", "Longitude", "Latitude"])
    .dropna(subset=["Longitude", "Latitude"])
    .drop_duplicates()
)
rest_names_locations.head()

### Stage 4: Apply the Function to Every Row

In [ ]:
# axis="columns" tells .apply() to pass one row at a time to the function.
rest_names_locations.apply(distance_from_CityTech_geodesic, axis="columns").head()

### Stage 5: Store the Results in a New Column

In [ ]:
rest_names_locations["distance_from_CityTech"] = rest_names_locations.apply(
    distance_from_CityTech_geodesic, axis="columns"
)
rest_names_locations.head()

### Stage 6: Filter and Sort the Results

In [ ]:
# Restaurants within half a mile of City Tech, sorted by distance
(
    rest_names_locations
    .query("distance_from_CityTech < 0.5")
    .sort_values("distance_from_CityTech")
)

---
# **[EXTENSION] Section 13: Plotting Beyond the Basics**

Pandas provides a convenient `.plot()` method that produces matplotlib figures from Series and DataFrames. The example below builds on the "Has Mice" example from Section 7 to produce a horizontal bar chart of the most affected restaurants in ZIP code 11201.

Richer styling, multi-panel figures, and statistical visualizations using matplotlib and seaborn are covered in a later session.

In [ ]:
# Build a working DataFrame: "Has Mice" violations in ZIP 11201
has_mice_11201 = (
    restaurants
    .query(' `VIOLATION CODE` == "04L" & ZIPCODE == "11201" ')
    .filter(items=["DBA", "BUILDING", "STREET", "INSPECTION DATE"])
)
has_mice_11201.head()

>### Reading the Plot Code Below
>
>The chained expression below produces a horizontal bar chart in three steps:
>
>* `has_mice_11201["DBA"].value_counts()` counts how many times each restaurant name appears.
>* `[30::-1]` selects the top 31 entries and reverses their order, so that the largest count ends up at the top of the chart.
>* `.plot(kind="barh")` renders the result as a horizontal bar chart.

In [ ]:
has_mice_11201["DBA"].value_counts()[30::-1].plot(
    kind="barh",
    title='"Has Mice" Violations in ZIP 11201, by Restaurant Name'
)

---
# **Glossary**

| Term | Meaning |
|:---|:---|
| **DataFrame** | A two-dimensional labeled data structure (rows and columns) that is the central object in pandas. |
| **Series** | A one-dimensional labeled array; each column of a DataFrame is a Series. |
| **Index** | The set of labels assigned to the rows (or columns) of a DataFrame or Series. May be integers, strings, or datetimes. |
| **dtype** | The data type of a column (for example, `int64`, `float64`, `object`, `datetime64[ns]`, `category`). |
| **object dtype** | The pandas dtype used for columns containing strings or mixed types. |
| **Categorical** | A pandas dtype for columns with a fixed set of valid values (for example, restaurant grades A/B/C). Uses less memory and produces cleaner output than equivalent string columns. |
| **NaN** | "Not a Number"; pandas's representation of a missing value in numeric columns. Follows IEEE 754 semantics, including the property that `NaN != NaN`. |
| **NaT** | "Not a Time"; pandas's representation of a missing value in datetime columns. |
| **Vectorized operation** | An operation applied to a whole column at once rather than to individual values in a loop. Vectorized operations are typically much faster than equivalent loops. |
| **Boolean indexing** | Selecting rows of a DataFrame using a boolean Series of the same length. Rows where the boolean is `True` are kept. |
| **Chain notation** | The practice of linking multiple pandas method calls in a single parenthesized expression so that each operates on the result of the previous one. |
| **Projection** | Selecting a subset of columns from a DataFrame (the `SELECT col1, col2` operation in SQL). |
| **Selection** | Selecting a subset of rows from a DataFrame based on a condition (the `WHERE` clause in SQL). |
| **Aggregation** | Collapsing many values into a summary value such as a sum, mean, count, or unique-count. |
| **Grouping** | Splitting a DataFrame into subsets by the value of one or more columns, then aggregating each subset independently (the `GROUP BY` clause in SQL). |
| **Pivot table** | A two-dimensional aggregation that reshapes a DataFrame so that one column's values become the rows, another column's values become the columns, and a third column's values are aggregated in each cell. |
| **Resampling** | The time-series counterpart of grouping: regrouping a DataFrame with a datetime index into fixed-width time intervals such as days, weeks, or months. |